# 03 - Data Cleaning
## NYC Taxi Revenue Optimization Project

**Goal:** Apply all cleaning rules identified in `02_eda.sql` to produce
a reliable, analysis-ready dataset.

**Source Table:** `workspace.nyc_taxi.yellow_trips_raw` (248M trips)  
**Output Table:** `workspace.nyc_taxi.yellow_trips_cleaned`

**Cleaning Rules Applied:**
1. Remove Vendor 5 and 6 — unreliable vendors
2. Remove fare_amount outside $2.50–$500
3. Remove total_amount outside $2.50–$600
4. Remove tip_amount below $0 or above $200
5. Remove trip_distance outside 0.1–100 miles
6. Remove trip_minutes outside 1–300 minutes
7. Remove wrong year trips
8. Treat structural nulls as zero (congestion, airport fee, cbd fee)

**Every rule here is backed by EDA findings — nothing is guessed.**

### Step 1 — Load Raw Table

We load directly from the Delta table registered in `01_data_ingestion.py`.
No file paths — Delta handles everything.

We select only the columns we need for analysis.
Dropping unused columns now keeps the cleaned table lean and fast.

In [0]:
from pyspark.sql.functions import (
    col, lit, coalesce, unix_timestamp, round, when, year, isnan, count, sum as spark_sum
)
from pyspark.sql.types import DoubleType

# Load only columns we need — drop irrelevant ones identified in EDA
df = spark.table("workspace.nyc_taxi.yellow_trips_raw").select(
    "VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "passenger_count",
    "trip_distance",
    "RatecodeID",
    "fare_amount",
    "tip_amount",
    "tolls_amount",
    "total_amount",
    "congestion_surcharge",
    "airport_fee",
    "cbd_congestion_fee",
    "payment_type",
    "data_year",
    "data_month"
)

raw_count = df.count()
print(f"✅ Raw table loaded")
print(f"   Columns selected: {len(df.columns)}")
print(f"   Total rows: {raw_count:,}")

### Step 2 — Add Derived Columns

Before filtering, we compute columns that:
1. Are needed for cleaning filters (trip_minutes)
2. Will be used directly in revenue analysis (revenue_per_mile, revenue_per_minute)

**Why add these now?**
Computing them on the full raw table means we only do this once.
The cleaned table will have everything ready for analysis — no recomputation needed.

**Columns Added:**
- `trip_minutes` — duration from pickup to dropoff
- `trip_speed_mph` — distance / duration, catches impossible speeds
- `revenue_per_mile` — total amount / distance
- `revenue_per_minute` — total amount / duration
- `tip_pct` — tip as % of fare
- `pickup_hour` — hour extracted from pickup datetime
- `pickup_day_of_week` — day name from pickup datetime
- `is_airport_trip` — flag for JFK, LaGuardia, Newark pickups

In [0]:
from pyspark.sql.functions import (
    round, unix_timestamp, hour, dayofweek,
    date_format, when, col
)

df = df \
    .withColumn("trip_minutes",
        round((unix_timestamp("tpep_dropoff_datetime")
             - unix_timestamp("tpep_pickup_datetime")) / 60.0, 2)) \
    .withColumn("trip_speed_mph",
        round(col("trip_distance") /
            (col("trip_minutes") / 60.0), 2)) \
    .withColumn("revenue_per_mile",
        round(col("total_amount") /
            col("trip_distance"), 2)) \
    .withColumn("revenue_per_minute",
        round(col("total_amount") /
            col("trip_minutes"), 2)) \
    .withColumn("tip_pct",
        round(col("tip_amount") /
            col("fare_amount") * 100.0, 2)) \
    .withColumn("pickup_hour",
        hour("tpep_pickup_datetime")) \
    .withColumn("pickup_day_of_week",
        date_format("tpep_pickup_datetime", "EEEE")) \
    .withColumn("is_airport_trip",
        when(col("PULocationID").isin(1, 132, 138), 1).otherwise(0))

print(f"✅ Derived columns added")
print(f"   Total columns now: {len(df.columns)}")
print(f"   New columns: trip_minutes, trip_speed_mph, revenue_per_mile,")
print(f"                revenue_per_minute, tip_pct, pickup_hour,")
print(f"                pickup_day_of_week, is_airport_trip")

### Step 3 — Apply Cleaning Filters

We apply all 8 cleaning rules identified in EDA.
Each rule is a separate filter so we can measure row impact individually.

**Important:** We apply filters sequentially and track rows removed
at each step — this is called a cleaning audit trail.
It tells us exactly which rule removed how many rows
and confirms no single rule is being too aggressive.

In [0]:
df_clean = df.filter(
    (~col("VendorID").isin(5, 6))                        &
    (col("fare_amount").between(2.50, 500))              &
    (col("total_amount").between(2.50, 600))             &
    (col("tip_amount").between(0, 200))                  &
    (col("trip_distance").between(0.1, 100))             &
    (col("trip_minutes").between(1, 300))                &
    (year("tpep_pickup_datetime") == col("data_year"))
) \
.withColumn("congestion_surcharge",
    coalesce(col("congestion_surcharge"), lit(0))) \
.withColumn("airport_fee",
    coalesce(col("airport_fee"), lit(0))) \
.withColumn("cbd_congestion_fee",
    coalesce(col("cbd_congestion_fee"), lit(0)))

clean_count = df_clean.count()
removed = raw_count - clean_count
kept_pct = (clean_count / raw_count) * 100

print(f"Raw:     {raw_count:,}")
print(f"Clean:   {clean_count:,}")
print(f"Removed: {removed:,}")
print(f"Kept:    {kept_pct:.2f}%")

### Step 4 — Write Cleaned Delta Table

We write the cleaned DataFrame to a new Delta table.
Raw table stays untouched — we never overwrite source data.

Partitioned by `data_year` — same as raw table for consistency.

In [0]:
(df_clean
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("data_year")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.nyc_taxi.yellow_trips_cleaned")
)

print("✅ Cleaned Delta table written: workspace.nyc_taxi.yellow_trips_cleaned")

### Step 5 — Verify Cleaned Table

Final checks before closing the cleaning notebook:
1. Row count per year matches expectations
2. No nulls remaining on financial columns
3. All derived columns present and populated

In [0]:
from pyspark.sql.functions import count

spark.table("workspace.nyc_taxi.yellow_trips_cleaned") \
    .groupBy("data_year") \
    .count() \
    .orderBy("data_year") \
    .show()

### Step 6 — Post-Clean Null Check

Confirm zero nulls on all financial and location columns
after cleaning rules were applied.

In [0]:
key_cols = [
    "fare_amount", "tip_amount", "total_amount",
    "trip_distance", "trip_minutes", "PULocationID",
    "DOLocationID", "tpep_pickup_datetime",
    "congestion_surcharge", "airport_fee", "cbd_congestion_fee"
]

null_checks = df_clean.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in key_cols
])

null_checks.show(vertical=True)

### Step 7 — Derived Column Sanity Check

Derived columns like `revenue_per_mile` and `trip_speed_mph` involve
division — which can produce infinity or NaN if the denominator is zero.

We confirm no infinite or NaN values exist before writing to Delta.

In [0]:
derived_cols = [
    "trip_minutes", "trip_speed_mph",
    "revenue_per_mile", "revenue_per_minute", "tip_pct"
]

# Check for NaN and infinity using SQL expressions
checks = df_clean.select([
    spark_sum((col(c) == float('inf')).cast("int")).alias(f"inf_{c}")
    for c in derived_cols
] + [
    spark_sum(isnan(col(c)).cast("int")).alias(f"nan_{c}")
    for c in derived_cols
])

checks.show(vertical=True)